In [2]:
!pip install -q openai pinecone pypdf tiktoken


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import re
from pathlib import Path
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
from pypdf import PdfReader

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False

# API Keys
os.environ["OPENAI_API_KEY"] = ""
os.environ["PINECONE_API_KEY"] = ""

client = OpenAI()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [9]:
index_name = "rag-demo"

# Ensure the index is created with the correct dimension
if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)

pc.create_index(
    name=index_name,
    dimension=1536,  # for text-embedding-3-small
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

index = pc.Index(index_name)

In [11]:
if IN_COLAB:
    uploaded = files.upload()
    document_path = list(uploaded.keys())[0]
else:
    # Update this path when running in Jupyter/VS Code locally.
    document_path = r"./doc_aveng.txt"
    if not Path(document_path).exists():
        raise FileNotFoundError(f"File not found: {document_path}")
    print(f"Using local file: {document_path}")

Using local file: ./doc_aveng.txt


In [12]:
def load_document(file_path):
    suffix = Path(file_path).suffix.lower()

    if suffix == ".pdf":
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            content = page.extract_text()
            if content:
                text += content + "\n"
        return text

    if suffix == ".txt":
        return Path(file_path).read_text(encoding="utf-8")

    raise ValueError("Only .pdf and .txt files are supported")

def normalize_text(text):
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def split_sentences(text):
    cleaned = re.sub(r"\s+", " ", text).strip()
    if not cleaned:
        return []
    return [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+", cleaned) if sentence.strip()]

def fixed_chunks(text, chunk_size=800, overlap=100):
    chunks = []
    start = 0
    text = normalize_text(text)
    step = max(chunk_size - overlap, 1)
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += step
    return chunks

def paragraph_chunks(text, chunk_size=800, overlap=100):
    paragraphs = [paragraph.strip() for paragraph in re.split(r"\n\s*\n", normalize_text(text)) if paragraph.strip()]
    chunks = []
    current = ""

    for paragraph in paragraphs:
        candidate = f"{current}\n\n{paragraph}".strip() if current else paragraph
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                chunks.append(current)
            current = paragraph

    if current:
        chunks.append(current)

    return chunks or fixed_chunks(text, chunk_size=chunk_size, overlap=overlap)

def sentence_chunks(text, chunk_size=800, overlap=100):
    sentences = split_sentences(text)
    chunks = []
    current = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence)
        if current and current_length + sentence_length + 1 > chunk_size:
            chunks.append(" ".join(current))
            if overlap > 0:
                overlap_text = chunks[-1][-overlap:].strip()
                current = [overlap_text, sentence] if overlap_text else [sentence]
                current_length = len(" ".join(current))
            else:
                current = [sentence]
                current_length = sentence_length
        else:
            current.append(sentence)
            current_length += sentence_length + (1 if current_length else 0)

    if current:
        chunks.append(" ".join(current).strip())

    return [chunk for chunk in chunks if chunk]

def recursive_chunks(text, chunk_size=800, overlap=100, separators=None):
    text = normalize_text(text)
    separators = separators or ["\n\n", "\n", ". ", " ", ""]

    def split_recursively(segment, remaining_separators):
        segment = segment.strip()
        if not segment:
            return []
        if len(segment) <= chunk_size:
            return [segment]
        if not remaining_separators:
            return fixed_chunks(segment, chunk_size=chunk_size, overlap=overlap)

        separator = remaining_separators[0]
        if separator == "":
            return fixed_chunks(segment, chunk_size=chunk_size, overlap=overlap)

        parts = segment.split(separator)
        if len(parts) == 1:
            return split_recursively(segment, remaining_separators[1:])

        merged_parts = []
        current = ""
        for part in parts:
            part = part.strip()
            if not part:
                continue
            candidate = f"{current}{separator}{part}".strip() if current else part
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    merged_parts.extend(split_recursively(current, remaining_separators[1:]))
                current = part

        if current:
            merged_parts.extend(split_recursively(current, remaining_separators[1:]))

        return merged_parts

    return split_recursively(text, separators)

def cosine_similarity(vec_a, vec_b):
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    magnitude_a = sum(a * a for a in vec_a) ** 0.5
    magnitude_b = sum(b * b for b in vec_b) ** 0.5
    if not magnitude_a or not magnitude_b:
        return 0
    return dot_product / (magnitude_a * magnitude_b)

def semantic_chunks(text, chunk_size=800, similarity_threshold=0.82):
    sentences = split_sentences(text)
    if not sentences:
        return []

    sentence_embeddings = [get_embedding(sentence) for sentence in sentences]
    chunks = []
    current_sentences = [sentences[0]]
    current_length = len(sentences[0])

    for idx in range(1, len(sentences)):
        sentence = sentences[idx]
        similarity = cosine_similarity(sentence_embeddings[idx - 1], sentence_embeddings[idx])
        would_exceed = current_length + len(sentence) + 1 > chunk_size

        if would_exceed or similarity < similarity_threshold:
            chunks.append(" ".join(current_sentences).strip())
            current_sentences = [sentence]
            current_length = len(sentence)
        else:
            current_sentences.append(sentence)
            current_length += len(sentence) + 1

    if current_sentences:
        chunks.append(" ".join(current_sentences).strip())

    return [chunk for chunk in chunks if chunk]

def chunk_text(text, strategy="recursive", chunk_size=800, overlap=100, similarity_threshold=0.82):
    strategies = {
        "recursive": lambda value: recursive_chunks(value, chunk_size=chunk_size, overlap=overlap),
        "sentence": lambda value: sentence_chunks(value, chunk_size=chunk_size, overlap=overlap),
        "paragraph": lambda value: paragraph_chunks(value, chunk_size=chunk_size, overlap=overlap),
        "fixed": lambda value: fixed_chunks(value, chunk_size=chunk_size, overlap=overlap),
        "semantic": lambda value: semantic_chunks(value, chunk_size=chunk_size, similarity_threshold=similarity_threshold),
    }

    if strategy not in strategies:
        raise ValueError(f"Unsupported chunking strategy: {strategy}. Choose from {list(strategies)}")

    return strategies[strategy](text)

In [13]:
def get_embedding(text):
    return client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    ).data[0].embedding

In [14]:
text = load_document(document_path)

# Change this to: recursive, sentence, paragraph, fixed, or semantic
chunking_strategy = "recursive"
chunk_size = 800
overlap = 100
similarity_threshold = 0.82

chunks = chunk_text(
    text,
    strategy=chunking_strategy,
    chunk_size=chunk_size,
    overlap=overlap,
    similarity_threshold=similarity_threshold,
)

print(f"Using {chunking_strategy} chunking")
print(f"Total chunks created: {len(chunks)}")
if chunks:
    print("Preview of first chunk:\n")
    print(chunks[0][:500])

vectors = []
for i, chunk in enumerate(chunks):
    emb = get_embedding(chunk)
    vectors.append({
        "id": f"{chunking_strategy}-{i}",
        "values": emb,
        "metadata": {
            "text": chunk,
            "chunking_strategy": chunking_strategy
        }
    })

index.upsert(vectors=vectors)
print("Stored in Pinecone ✅")

Using recursive chunking
Total chunks created: 17
Preview of first chunk:

Having acquired the Power Stone from the planet Xandar, Thanos and his lieutenants- Ebony Maw, Cull Obsidian, Proxima Midnight and Corvus Glaive intercept the spaceship carrying the survivors of Asgard's destruction. Thanos destroys the ship and kills all surviving Asgardians.
Stored in Pinecone ✅


In [13]:
def retrieve(query, top_k=3):
    query_emb = get_embedding(query)

    results = index.query(
        vector=query_emb,
        top_k=top_k,
        include_metadata=True
    )

    docs = [match["metadata"]["text"] for match in results["matches"]]
    return docs

In [14]:
def stream_answer(query, docs):
    context = "\n\n".join(docs)

    prompt = f"""
Answer ONLY from the given context.

Context:
{context}

Question:
{query}
"""

    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        stream=True
    )

    full_text = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            token = chunk.choices[0].delta.content
            print(token, end="", flush=True)
            full_text += token

    print()
    return full_text

In [15]:
while True:
    q = input("Ask (type 'exit' to stop): ")
    if q.lower() == "exit":
        break

    docs = retrieve(q)
    print("Answer:")
    stream_answer(q, docs)

Ask (type 'exit' to stop): Tell me overview
Answer:
The context provided is from a training session on Large Language Models (LLMs) and their applications, specifically focusing on building summarization and translation agents. It includes multiple-choice questions about key characteristics of LLMs, the concept of a context window, handling API rate limits, and considerations for choosing between different models for summarization tasks. The session emphasizes the importance of understanding token limits, output consistency, and the specific needs of tasks like summarization and translation.
Ask (type 'exit' to stop): exit
